# Limpieza de datos

Se descargaron los archivos de los resultados desde 2021 hasta la temporada actual desde football-data: https://football-data.co.uk/englandm.php . 
Despues se seleccionaron aquellas variables dentro de todos los dataframes (columnas obligatorias), y columnas complementarias relacionadas a apuestas, que nos servirán para hacer comparaciones con probabilidades implicitas de estas y los resultados de nuestro predictor. El código siguiente tiene como resultado un csv que adjunta los archivos csv descargados y crea un histórico con las variables "columnas obligatorias" y "columnas complementarias debajo descritas"

In [ ]:
import pandas as pd
# El siguiente codigo asume que los csvs descargados estan en la carpeta de descargas y que tienen el nombre E0.csv, E0 (1).csv, E0 (2).csv, ..., E0 (25).csv
columnas_obligatorias = [
    "Date",
    "HomeTeam",
    "AwayTeam",
    "FTHG",
    "FTAG",
    "FTR",
    "HTHG",
    "HTAG",
    "HTR",
    "HS",
    "AS",
    "HST",
    "AST",
    "HC",
    "AC",
    "HF",
    "AF",
    "HY",
    "AY",
    "HR",
    "AR",
    "Referee",
]
#Estas no están completas para todos los años pero sirven para analisis estadistico de aquellas que si los tienen
columnas_complementarias = [
    "B365H",
    "B365D",
    "B365A",
    "AvgH",
    "AvgD",
    "AvgA",
    "AvgCH",
    "AvgCD",
    "AvgCA",
    "HxG",
    "AxG",
]

# Lista total de columnas
columnas_totales = columnas_obligatorias + columnas_complementarias

# Rutas de los archivos (E0.csv el primero  y concatena de E0 (1).csv a E0 (25))
                       #nombre de tu usuario
archivos = [r"C:\Users\Daniel\Downloads\E0.csv"] + [
    rf"C:\Users\Daniel\Downloads\E0 ({i}).csv" for i in range(1, 26)
]

lista_dfs = []

for idx, ruta in enumerate(archivos):
    nombre_archivo = ruta.split("\\")[-1]
    try:
        df_temp = pd.read_csv(
            ruta, encoding="latin1", on_bad_lines="skip", engine="python"
        )

        # Normalizar los nombres de columnas (elimina espacios extra)
        df_temp.columns = df_temp.columns.str.strip()

        # Reindexar asegura que conservemos exactamente las columnas solicitadas.
        # Las columnas que no existan en el archivo actual se rellenarán con NaN (vacías).
        df_filtrado = df_temp.reindex(columns=columnas_totales)

        lista_dfs.append(df_filtrado)

    except FileNotFoundError:
        print(f"{nombre_archivo}: No se encontró el archivo (omitido).")
    except Exception as e:
        print(f"{nombre_archivo}: Error al procesar: {e}")

# Concatenar y guardar
if lista_dfs:
    df_resultado = pd.concat(lista_dfs, ignore_index=True)

    # Eliminar filas completamente vacías si existieran al final de los archivos
    df_resultado.dropna(subset=["HomeTeam", "AwayTeam"], how="all", inplace=True)
    df_resultado.sort_values(by="Date", inplace=True, ascending=False)

    # Convertimos la columna Date a tipo fecha
    df_resultado['Date'] = pd.to_datetime(
        df_resultado['Date'], dayfirst=True, format='mixed', errors='coerce'
    )

    # Ordenar de forma cronológica por fecha empezando por la más reciente
    df_resultado.sort_values(by='Date', inplace=True,ascending=False)

    # Formatear la fecha a YYYY-MM-DD para la exportación a CSV
    df_resultado['Date'] = df_resultado['Date'].dt.strftime('%Y-%m-%d')

    ruta_salida = r"C:\Users\Daniel\Downloads\E0_filtrado_consolidado.csv"
#    df_resultado.to_csv(ruta_salida, index=False, encoding="utf-8-sig")

    print("Done")
    print(f"Archivo guardado en \n{ruta_salida}")
else:
    print("No se procesó ningún archivo.")

Done
Archivo guardado en:
C:\Users\roski\Downloads\E0_filtrado_consolidado.csv
